# NB06 — Clean SAM recovery | Baego + Luki
**Aim:** inspect archived automatic SAM1/SAM3 runs and compare saved crown polygons with reference crowns. **No training or inference. No historical outputs modified.** The historical six-ROI SAM3 F1 ≈ 0.490 is a reference, **not** automatically reproduced here.

Run blocks in order. Select archived outputs in Block 04 only after reviewing filenames and metadata.

In [ ]:
# 01 — Setup (read-only paths)
from pathlib import Path
from google.colab import drive
import json, re, warnings
import numpy as np, pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import box
from rasterio.windows import from_bounds
from rasterio.windows import transform as window_transform

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Colab Notebooks/Musanga')
DATA = ROOT / 'data/Geo_Ref'
ARCHIVE = ROOT / 'KVO_work/KVO_06_UNet_guided_SAM3'
ARCHIVE_V2 = ROOT / 'KVO_work/KVO_06_v2_Baego_Luki'
INPUTS = {'Baego_RGB': DATA/'Baego_georef.tif', 'Baego_Kristof': DATA/'Annotations/Crown_Kristof.shp', 'Baego_Nadine': DATA/'Annotations/Crown_Nadine.shp', 'Baego_ROI': DATA/'Annotations/ROI_Crowns_Kvo.shp', 'Luki_RGB': DATA/'Luki.tif', 'Luki_reference': DATA/'Annotations/Tree_Annotation_Luki.gpkg'}
for name,p in INPUTS.items(): print(f'{name:18} {"OK" if p.exists() else "MISSING"}  {p}')
print('Historical folders:', [(str(p),p.exists()) for p in (ARCHIVE,ARCHIVE_V2)])

In [ ]:
# 02 — Reference crowns and area rule (metres squared)
MIN_CROWN_M2 = 5.0

def load_layer(path, target_crs, raster_bounds, layer=None):
    g = gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)
    g = g.loc[g.geometry.notna() & ~g.geometry.is_empty].copy()
    if g.empty:
        print(f'WARNING: no geometries in {path.name}')
        return g

    # A missing .prj is common in older annotation shapefiles.
    # Assign raster CRS only when the raw coordinate extents already overlap.
    if g.crs is None:
        xmin, ymin, xmax, ymax = g.total_bounds
        rxmin, rymin, rxmax, rymax = tuple(raster_bounds)
        overlaps = xmax >= rxmin and xmin <= rxmax and ymax >= rymin and ymin <= rymax
        if not overlaps:
            raise ValueError(
                f'{path.name} has no CRS and its coordinates do not overlap the RGB raster. '
                f'Annotation bounds={g.total_bounds}; raster bounds={tuple(raster_bounds)}. '
                'Check the original annotation CRS before proceeding.'
            )
        print(f'WARNING: {path.name} has no CRS; coordinates overlap raster. '
              f'Assuming raster CRS {target_crs}.')
        g = g.set_crs(target_crs, allow_override=True)
    else:
        g = g.to_crs(target_crs)
    return g

def load_crowns(path, target_crs, raster_bounds, layer=None):
    g = load_layer(path, target_crs, raster_bounds, layer)
    if not g.crs.is_projected:
        raise ValueError('A projected metric CRS is required for the 5 m² crown filter')
    if g.crs.axis_info and not all(
        axis.unit_name.lower() in ('metre','meter','metres','meters') for axis in g.crs.axis_info[:2]
    ):
        raise ValueError(f'CRS units are not metres: {g.crs}')
    before = len(g)
    g = g.loc[g.geometry.area >= MIN_CROWN_M2].copy()
    print(f'  {path.name}: {before} polygons → {len(g)} crowns ≥ {MIN_CROWN_M2:g} m²')
    return g

SITES = {}
for site, rgb_key in [('Baego','Baego_RGB'),('Luki','Luki_RGB')]:
    if not INPUTS[rgb_key].exists(): continue
    with rasterio.open(INPUTS[rgb_key]) as ds:
        if ds.crs is None or not ds.crs.is_projected:
            raise ValueError(f'{site}: projected raster CRS required')
        site_info = {'rgb':INPUTS[rgb_key], 'crs':ds.crs,
                     'bounds':ds.bounds, 'res':ds.res}
    crs, bounds = site_info['crs'], site_info['bounds']
    print(f'\n{site}: raster CRS={crs}, resolution={site_info["res"]}')
    if site == 'Baego':
        site_info['Kristof'] = load_crowns(INPUTS['Baego_Kristof'], crs, bounds)
        site_info['Nadine'] = load_crowns(INPUTS['Baego_Nadine'], crs, bounds)
        site_info['roi'] = load_layer(INPUTS['Baego_ROI'], crs, bounds)
    else:
        site_info['reference'] = load_crowns(
            INPUTS['Luki_reference'], crs, bounds,
            layer='photointerpretation_inventory')
    SITES[site] = site_info
    print('Reference counts:', {
        k:len(v) for k,v in site_info.items()
        if isinstance(v,gpd.GeoDataFrame)
    })

In [ ]:
# 03 — Inspect archive files + original SAM configuration (do not execute archived code)
# This is a text inspection, not model initialization.
SEARCH_ROOTS = [p for p in (ROOT/'KVO_work',ROOT) if p.exists()]
SEARCH_ROOT = SEARCH_ROOTS[0]
notebooks = sorted(SEARCH_ROOT.rglob('*.ipynb'))
print('SAM notebooks (up to 60):')
for p in [p for p in notebooks if any(x in p.name.lower() for x in ('sam','06','crown'))][:60]: print('  ',p.relative_to(ROOT))
print('\nLikely historical configs:')
PAT = re.compile(r'(points_per_batch|points_per_side|points_per_crop|crops_n_layers|crop_overlap_ratio|pred_iou_thresh|stability_score_thresh|stability_score_offset|mask_threshold|max_hole_area|max_sprinkle_area|tile_size|tile_px|overlap|context|sam1|sam3)',re.I)
for p in [p for p in notebooks if ('sam' in p.name.lower() or '06' in p.name.lower())][:30]:
    try:
        obj=json.loads(p.read_text(errors='replace'))
        hits=[]
        for i,c in enumerate(obj.get('cells',[])):
            src=''.join(c.get('source',[]))
            for line in src.splitlines():
                if PAT.search(line) and len(line)<220 and not line.lstrip().startswith('#'):
                    hits.append((i,line.strip()))
        if hits:
            print('\n',p.relative_to(ROOT))
            for i,line in hits[:25]: print(f'  cell {i}: {line[:180]}')
    except Exception as e: print('Cannot inspect',p.name,str(e)[:100])

In [ ]:
# 04 — Load the historical SAM1/SAM3 comparison automatically (no inference)
# This is the Oct 9 v58 comparison, NOT the older six-ROI F1≈0.490 baseline.
HISTORY = ARCHIVE / 'experiments_v58_full_pretest_sam1_sam3'
if not HISTORY.exists():
    raise FileNotFoundError(f'Historical comparison not found: {HISTORY}')

csv_path = HISTORY / 'five_roi_sam1_sam3_comparison.csv'
if csv_path.exists():
    METRICS = pd.read_csv(csv_path)
    print('=== SAVED SAM1/SAM3 RESULTS (v58; not the six-ROI baseline) ===')
    display(METRICS.head(20))
else:
    print('No comparison CSV at:',csv_path)

PREDICTIONS_BY_ROI = {}
for roi_number in range(5):
    for method in ('SAM1_ViTH', 'SAM3_permissive'):
        p = HISTORY / f'roi{roi_number}_{method}_crowns.gpkg'
        if not p.exists(): continue
        try:
            g = gpd.read_file(p)
            if g.crs is None:
                print('SKIP — missing polygon CRS:',p.name)
                continue
            g = g.to_crs(SITES['Baego']['crs'])
            PREDICTIONS_BY_ROI[(roi_number,method)] = g
        except Exception as exc:
            print('SKIP',p.name, str(exc)[:130])
print('\nLoaded archived Baego ROI/method combinations:',sorted(PREDICTIONS_BY_ROI))
print('Luki: historical comparison folder:', ARCHIVE/'experiments_v58b_luki_context_sam1_sam3')
print('No SAM models loaded or executed.')

In [ ]:
# 05 — Direct visual assessment: RGB + original crowns + saved SAM1/SAM3
# These historical polygon files use roi0..roi4. Check visual spatial overlap.
VIEW_ROI = 1  # change to 0,1,2,3,4
SITE='Baego'
s=SITES[SITE]
roi=s['roi']
if VIEW_ROI >= len(roi):
    raise ValueError(f'ROI {VIEW_ROI} out of bounds (only {len(roi)} source ROIs)')
bounds=roi.geometry.iloc[VIEW_ROI].bounds
fig,axs=plt.subplots(1,2,figsize=(16,8),constrained_layout=True)
with rasterio.open(s['rgb']) as ds:
    w=from_bounds(*bounds,transform=ds.transform).round_offsets().round_lengths()
    from rasterio.windows import Window
    w=w.intersection(Window(0,0,ds.width,ds.height))
    rgb=ds.read([1,2,3],window=w,out_shape=(3,min(int(w.height),1400),min(int(w.width),1400)))
    bb=rasterio.windows.bounds(w,ds.transform)
img=np.moveaxis(rgb,0,-1).astype(float)
lo,hi=np.nanpercentile(img,[2,98]);img=np.clip((img-lo)/(hi-lo+1e-9),0,1)
extent=[bb[0],bb[2],bb[1],bb[3]]
roi_box=box(*bb)
for ax,method in zip(axs,('SAM1_ViTH','SAM3_permissive')):
    ax.imshow(img,extent=extent)
    for label,color in [('Kristof','white'),('Nadine','yellow')]:
        g=s[label];g=g[g.geometry.intersects(roi_box)]
        if len(g):g.boundary.plot(ax=ax,color=color,linewidth=.9)
    pred=PREDICTIONS_BY_ROI.get((VIEW_ROI,method))
    if pred is not None:
        visible=pred[pred.geometry.intersects(roi_box)]
        if len(visible):visible.boundary.plot(ax=ax,color='red',linewidth=.9)
        ax.set_title(f'{method}: {len(visible)} saved polygons')
    else:
        ax.set_title(method+' — no saved polygon file')
    ax.set_xlim(bb[0],bb[2]);ax.set_ylim(bb[1],bb[3]);ax.set_aspect('equal')
fig.suptitle(f'Baego ROI {VIEW_ROI} | white=Kristof; yellow=Nadine; red=historical SAM')
plt.show()
print('To review another ROI, change VIEW_ROI in Block 05 and rerun ONLY that block.')
print('This shows v58 archived outputs, not necessarily the original six-ROI baseline.')

In [ ]:
# 06 — Interpretation: what is known and what still needs identification
print('HISTORICAL SIX-ROI SAM3 F1 ≈ 0.490 — reference only, not reproduced.')
print('v58 comparison polygon sets:',len(PREDICTIONS_BY_ROI))
if 'METRICS' in globals():
    print('v58 metrics file columns:',list(METRICS.columns))
print('Look at the red crown outlines in Block 05. Prioritize complete individual crowns.')
print('Next: recover the exact six-ROI baseline + original Luki configuration, not tune thresholds.')
print('No new training or inference was performed.')

## 07 — Decision log (fill after inspecting original archive)
- **Original notebook and run ID:** not yet verified
- **SAM1 checkpoint / SAM3 model revision:** not yet verified
- **Tiling dimensions / overlap / context:** not yet verified
- **Prompting mode:** not yet verified
- **AMG thresholds:** not yet verified (do not confuse Hugging Face defaults with actual run settings)
- **Postprocessing / overlap handling:** not yet verified
- **Six historical ROI identifiers and saved output:** not yet verified
- **Historical SAM3 F1 ≈ 0.490:** preserved as reported benchmark only

**Decision gate:** only consider a new inference block after the archived run and its visual outputs have been positively identified.